In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Concatenate, Layer, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

# --- Custom Gating Layer (Corrected) --- 
class AttentionGatingLayer(Layer):
    """
    A custom layer that performs attention-based fusion of two embeddings.
    It first projects the embeddings to a shared dimension to allow for fusion.
    """
    def __init__(self, output_dim, **kwargs):
        super(AttentionGatingLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
        
    def build(self, input_shape):
        gnn_shape, llm_shape = input_shape
        
        # Project GNN embedding to a shared dimension
        self.gnn_project = Dense(self.output_dim, activation='relu', name='gnn_project')
        
        # Project LLM embedding to the same shared dimension
        self.llm_project = Dense(self.output_dim, activation='relu', name='llm_project')
        
        # Gating layers now operate on the projected embeddings
        self.gnn_gate = Dense(self.output_dim, activation='sigmoid', name='gnn_gate')
        self.llm_gate = Dense(self.output_dim, activation='sigmoid', name='llm_gate')
        
        # Alpha layer to calculate the fusion weight
        self.alpha_layer = Dense(1, activation='sigmoid', name='alpha_layer')
        
        super(AttentionGatingLayer, self).build(input_shape)

    def call(self, inputs):
        gnn_embedding, llm_embedding = inputs
        
        # Project both embeddings to the same dimension before fusion
        gnn_projected = self.gnn_project(gnn_embedding)
        llm_projected = self.llm_project(llm_embedding)
        
        # Apply gating to the projected embeddings
        gnn_gated = self.gnn_gate(gnn_projected)
        llm_gated = self.llm_gate(llm_projected)
        
        # The sum now works because both tensors have the same shape
        combined = gnn_gated + llm_gated
        
        # Calculate the attention weight
        alpha = self.alpha_layer(combined)
        
        # Fuse the embeddings
        fused_embedding = alpha * gnn_projected + (1 - alpha) * llm_projected
        
        return fused_embedding

    def compute_output_shape(self, input_shape):
        return (input_shape[0][0], self.output_dim)

# --- Helper functions ---
def downcast_dataframe(df):
    """Downcasts numeric data types to save memory."""
    print("Downcasting numeric columns to save memory...")
    for col in df.select_dtypes(include=np.number).columns:
        df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
    return df

print("\n--- Starting Hybrid Model Training with Gating ---")

# --- Load and Sample Data ---
print("Loading sampled transaction and identity data...")
try:
    X_transaction_df = pd.read_csv("../data/processed/train_transaction_sample.csv").sample(frac=0.1, random_state=42)
    X_identity_df = pd.read_csv("../data/processed/train_identity_sample.csv")
    y = X_transaction_df['isFraud'].values
except FileNotFoundError as e:
    print(f"Error: {e}. Please run your EDA script first to generate sample files.")
    exit()

# Downcast initial dataframes to save memory
X_transaction_df = downcast_dataframe(X_transaction_df)
X_identity_df = downcast_dataframe(X_identity_df)

# Merge the transaction and identity dataframes
print("Merging transaction and identity dataframes...")
X_df = X_transaction_df.merge(X_identity_df, on="TransactionID", how="left")
print(f"Shape of the merged dataframe: {X_df.shape}")
transaction_ids = X_df['TransactionID'].values

# --- Load Embeddings and Merge (Filtered) ---
llm_embeddings_path = "../data/processed/llm_embeddings.csv"
gnn_path = "../data/processed/gnn_embeddings.csv"

# Handle potential FileNotFoundError and ensure data types are correct
if os.path.exists(llm_embeddings_path):
    llm_embeddings_full = pd.read_csv(llm_embeddings_path, low_memory=False)
    llm_embeddings_full = downcast_dataframe(llm_embeddings_full)
    llm_embeddings = llm_embeddings_full[llm_embeddings_full['TransactionID'].isin(transaction_ids)]
    del llm_embeddings_full
    print(f"LLM embeddings loaded and filtered. Shape: {llm_embeddings.shape}")
    X_df = X_df.merge(llm_embeddings, on="TransactionID", how='left')
else:
    print(f"Warning: {llm_embeddings_path} not found. Skipping LLM embeddings.")
    
if os.path.exists(gnn_path):
    gnn_full = pd.read_csv(gnn_path, low_memory=False)
    gnn_full = downcast_dataframe(gnn_full)
    gnn = gnn_full[gnn_full['TransactionID'].isin(transaction_ids)]
    del gnn_full
    print(f"GNN embeddings loaded and filtered. Shape: {gnn.shape}")
    X_df = X_df.merge(gnn, on="TransactionID", how='left')
else:
    print(f"Warning: {gnn_path} not found. Skipping GNN embeddings.")

# --- Prepare Features & Labels ---
X_df = X_df.drop(columns=['TransactionID', 'isFraud'], errors="ignore")
leaky_cols = ['DeviceInfo', 'card1', 'id_31', 'id_33', 'Prompt']
X_df = X_df.drop(columns=leaky_cols, errors='ignore')

HIGH_CARDINALITY_THRESHOLD = 50
categorical_cols_all = X_df.select_dtypes(include=['object', 'category']).columns.tolist()
high_cardinality_cols = [col for col in categorical_cols_all if X_df[col].nunique() > HIGH_CARDINALITY_THRESHOLD]
X_df = X_df.drop(columns=high_cardinality_cols, errors='ignore')
categorical_cols = [col for col in X_df.columns if col in categorical_cols_all and col not in high_cardinality_cols]

gnn_cols = [col for col in X_df.columns if col.startswith('gnn_embed_')]
llm_embed_cols = [col for col in X_df.columns if col.startswith('LLM_embed_')]
other_numeric_cols = [col for col in X_df.select_dtypes(include=np.number).columns.tolist() if col not in gnn_cols and col not in llm_embed_cols]

# --- Preprocessing Pipeline ---
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, other_numeric_cols),
        ('cat', categorical_transformer, categorical_cols)],
    remainder='passthrough'
)

#  Split the data and apply preprocessor
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

preprocessor.fit(X_train_df)
X_train_processed = preprocessor.transform(X_train_df).astype('float32')
X_test_processed = preprocessor.transform(X_test_df).astype('float32')

X_train_gnn = X_train_df[gnn_cols].replace(['', '{}'], np.nan).fillna(0.0).values.astype('float32')
X_test_gnn = X_test_df[gnn_cols].replace(['', '{}'], np.nan).fillna(0.0).values.astype('float32')
X_train_llm = X_train_df[llm_embed_cols].replace(['', '{}'], np.nan).fillna(0.0).values.astype('float32')
X_test_llm = X_test_df[llm_embed_cols].replace(['', '{}'], np.nan).fillna(0.0).values.astype('float32')

print("\nFitting and transforming data...")
print(f"Shape of X_train processed tabular data: {X_train_processed.shape}")
print(f"Shape of X_train GNN embeddings: {X_train_gnn.shape}")
print(f"Shape of X_train LLM embeddings: {X_train_llm.shape}")

#  Neural Network Model with Keras (Dynamic Building)
num_non_fraud = np.sum(y_train == 0)
num_fraud = np.sum(y_train == 1)
class_weight = {0: 1., 1: num_non_fraud / num_fraud}
print(f"Calculated class weights: {class_weight}")

inputs_list = []
concatenation_list = []

# Tabular input is always present
input_tabular = Input(shape=(X_train_processed.shape[1],), name='tabular_input')
inputs_list.append(input_tabular)
concatenation_list.append(input_tabular)

# Check and add GNN/LLM inputs if they exist
if gnn_cols and llm_embed_cols:
    print("Using Attention Gating Fusion Layer.")
    input_gnn = Input(shape=(X_train_gnn.shape[1],), name='gnn_input')
    input_llm = Input(shape=(X_train_llm.shape[1],), name='llm_input')
    inputs_list.append(input_gnn)
    inputs_list.append(input_llm)
    fused_embeddings = AttentionGatingLayer(output_dim=64, name='fusion_layer')([input_gnn, input_llm])
    concatenation_list.append(fused_embeddings)
elif gnn_cols:
    print("Using only GNN embeddings with concatenation.")
    input_gnn = Input(shape=(X_train_gnn.shape[1],), name='gnn_input')
    inputs_list.append(input_gnn)
    concatenation_list.append(input_gnn)
elif llm_embed_cols:
    print("Using only LLM embeddings with concatenation.")
    input_llm = Input(shape=(X_train_llm.shape[1],), name='llm_input')
    inputs_list.append(input_llm)
    concatenation_list.append(input_llm)
else:
    print("No GNN or LLM embeddings found. Using only tabular data.")

if len(concatenation_list) > 1:
    concatenated_features = Concatenate(name='final_concat')(concatenation_list)
else:
    concatenated_features = concatenation_list[0]

# Build the main model
x = Dense(256, activation='relu')(concatenated_features)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs_list, outputs=output)

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])

early_stopping = EarlyStopping(monitor='val_auc', patience=10, mode='max', restore_best_weights=True)

# Prepare inputs for the fit function
fit_inputs = {'tabular_input': X_train_processed}
val_inputs = {'tabular_input': X_test_processed}
if gnn_cols and X_train_gnn.shape[1] > 0:
    fit_inputs['gnn_input'] = X_train_gnn
    val_inputs['gnn_input'] = X_test_gnn
if llm_embed_cols and X_train_llm.shape[1] > 0:
    fit_inputs['llm_input'] = X_train_llm
    val_inputs['llm_input'] = X_test_llm

print("\nStarting Neural Network training...")
history = model.fit(
    fit_inputs,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(val_inputs, y_test),
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1
)

print("Neural Network training complete.")

# --- Evaluation ---
print("\n--- Model Evaluation ---")
probs = model.predict(val_inputs).flatten()
y_pred = (probs >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
conf_matrix = confusion_matrix(y_test, y_pred)
roc_auc = roc_auc_score(y_test, probs)

print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print("Confusion Matrix:")
print(conf_matrix)
print(f"ROC AUC Score: {roc_auc:.3f}")


--- Starting Hybrid Model Training with Gating ---
Loading sampled transaction and identity data...
Downcasting numeric columns to save memory...
Downcasting numeric columns to save memory...


C:\Users\aishu\AppData\Local\Temp\ipykernel_20484\3820999077.py:75: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
C:\Users\aishu\AppData\Local\Temp\ipykernel_20484\3820999077.py:75: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
C:\Users\aishu\AppData\Local\Temp\ipykernel_20484\3820999077.py:75: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
C:\Users\aishu\AppData\Local\Temp\ipykernel_20484\3820999077.py:75: FutureWarning: errors='ignore' is d

Merging transaction and identity dataframes...
Shape of the merged dataframe: (5000, 434)
Downcasting numeric columns to save memory...
LLM embeddings loaded and filtered. Shape: (95, 1538)


C:\Users\aishu\AppData\Local\Temp\ipykernel_20484\3820999077.py:75: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')


Downcasting numeric columns to save memory...
GNN embeddings loaded and filtered. Shape: (5000, 65)


C:\Users\aishu\AppData\Local\Temp\ipykernel_20484\3820999077.py:75: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
C:\Users\aishu\AppData\Local\Temp\ipykernel_20484\3820999077.py:75: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
C:\Users\aishu\AppData\Local\Temp\ipykernel_20484\3820999077.py:75: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
C:\Users\aishu\AppData\Local\Temp\ipykernel_20484\3820999077.py:75: FutureWarning: errors='ignore' is d


Fitting and transforming data...
Shape of X_train processed tabular data: (4000, 2169)
Shape of X_train GNN embeddings: (4000, 64)
Shape of X_train LLM embeddings: (4000, 1536)
Calculated class weights: {0: 1.0, 1: 24.974025974025974}
Using Attention Gating Fusion Layer.

Starting Neural Network training...
Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.5408 - auc: 0.5111 - loss: 2.1732 - precision: 0.0374 - recall: 0.4416 - val_accuracy: 0.9620 - val_auc: 0.5506 - val_loss: 0.2394 - val_precision: 1.0000 - val_recall: 0.0256
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6070 - auc: 0.6309 - loss: 1.3714 - precision: 0.0574 - recall: 0.5974 - val_accuracy: 0.8630 - val_auc: 0.8119 - val_loss: 0.5339 - val_precision: 0.1397 - val_recall: 0.4872
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6118 - auc: 0.6800 - loss: 1.2142 - precision: 0.0603 - recall: 0.6234 - val_accuracy: 0.9010 - val_auc: 0.8319 - val_loss: 0.3464 - val_